In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations
from pathlib import Path
from typing import Any
import numpy as np
from steps.master_tdt_analysis import TdtExperiment

import pyqtgraph as pg

In [3]:
# =============================================================================
# EXPERIMENT SETTINGS
# =============================================================================

EXPERIMENT_NAME = "20191204"
SAMPLE_DELAY = 5

EXPERIMENT_ROOT = (
        Path(r"D:\ImThera\data_raw")
        / EXPERIMENT_NAME
)

EXPERIMENT_LOG = (
        EXPERIMENT_ROOT
        / "Notes"
        / f"{EXPERIMENT_NAME}_Experimental_Log.xlsx"
)

OUTPUT_DIR = (
        Path(r"D:\ImThera\data_processed")
        / EXPERIMENT_NAME
)

STORES = [
    "RawE",
    "RawG",
]

RECORDING_CHANNEL_NAMES = [
    "LIFE 1",
    "LIFE 2",
    "LIFE 3",
    "LIFE 4",
    "EMG 1",
    "EMG 2",
    "EMG 3",
]

RECORDING_CHANNEL_TYPES = [
    "ENG",
    "ENG",
    "ENG",
    "ENG",
    "EMG",
    "EMG",
    "EMG",
]

REMOVE_CHANNELS: list[str | int] = ["RawG 4"]

FILTER_MEDIAN_LOWPASS = False
FILTER_MEDIAN_HIGHPASS = True
FILTER_GAUSSIAN_HIGHPASS = True
FILTER_POWERLINE = True
FILTER_POST_AVERAGE = False
TDT_CHUNK_SIZE = 2_000_000


In [4]:
# =============================================================================
# LOAD EXPERIMENT
# =============================================================================

def load_experiment() -> Any:
    """
    Load, curate, and prepare the experiment.
    """
    if not EXPERIMENT_ROOT.exists():
        raise FileNotFoundError(
            f"Experiment folder not found: {EXPERIMENT_ROOT}"
        )

    if not EXPERIMENT_LOG.exists():
        raise FileNotFoundError(
            f"Experimental log not found: {EXPERIMENT_LOG}"
        )

    experiment = TdtExperiment(
        experiment_name=EXPERIMENT_NAME,
        experiment_storage_path=str(
            EXPERIMENT_ROOT
        ),
        tdt_chunk_size=TDT_CHUNK_SIZE,
        stores=STORES,
        sample_delay=SAMPLE_DELAY,
    )

    experiment.curate_data(
        ch_names=RECORDING_CHANNEL_NAMES,
        ch_types=RECORDING_CHANNEL_TYPES,
        remove_channels=REMOVE_CHANNELS,
        filter_median_low=FILTER_MEDIAN_LOWPASS,
        filter_median=FILTER_MEDIAN_HIGHPASS,
        filter_gaussian_highpass=FILTER_GAUSSIAN_HIGHPASS,
        filter_powerline=FILTER_POWERLINE,
    )

    experiment.gather_ecap(
        experiment_log_path=str(
            EXPERIMENT_LOG
        ),
        filter_post_average=FILTER_POST_AVERAGE,
        plot_AUCs=False,
    )

    return experiment

In [5]:
import time
start_time = time.perf_counter()
experiment = load_experiment()
print(f"Execution time: {time.perf_counter() - start_time}")


Execution time: 5.878198699996574


In [6]:
result = experiment.data_ecap.epoch(
    condition="Intact",
    pulse_amplitude="max",
)

In [7]:
t1 = np.mean(result.array, axis=1)
t1.shape

(10, 7, 977)

In [8]:
print(f"Graph tasks for source: {len(experiment.data_ephys.array.dask):,}")
print(f"Graph tasks for individual epoch array: {len(experiment.data_ecap.dask_array((0,0))):,}")

Graph tasks for source: 6,298
Graph tasks for individual epoch array: 750


In [11]:
start_time = time.perf_counter()
mean_traces = experiment.data_ecap.compute_mean_traces()
print(f"EMean traces computed in: {time.perf_counter() - start_time}")

EMean traces computed in: 620.3246702000033


In [12]:
start = time.perf_counter()
mean_traces_again = experiment.data_ecap.compute_mean_traces()
print(f"Cached call: {time.perf_counter() - start:.6f} seconds")

print(mean_traces_again is experiment.data_ecap.mean_traces)

Cached call: 0.000067 seconds
True


In [9]:
result = experiment.data_ecap.epoch(
    condition="Intact",
    pulse_amplitude="max",
    recording_channels=["LIFE " + str(i) for i in range(1,5)],
)

In [10]:
result = experiment.data_ecap.epoch(
    condition="Intact",
    pulse_amplitude="max",
    channel="Channel 1",
#    recording_channels="LIFE 1",
)